# Clean gridgame main + survey data

This notebook reads all `gridgame_clean_*.csv` files in the same folder, combines them, and writes the cleaned outputs into a subfolder named `cleaned_behavior_survey/`:

- `cleaned_behavior_survey/main_phase.csv`: cleaned event-level rows where `phase == "main"`
- `cleaned_behavior_survey/survey_phase.csv`: one summary row per participant for NFC, BIS/BAS, FiveDCR, demographics, strategy, and final agent ranking
- `cleaned_behavior_survey/main_phase_summary.csv`: optional participant-level behavior summary from the main phase

Put this notebook in the same folder as the raw participant CSV files and run all cells.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_DIR = Path(".")
if not list(INPUT_DIR.glob("gridgame_clean_*.csv")):
    INPUT_DIR = Path("/mnt/data")

INPUT_PATTERN = "gridgame_clean_*.csv"
OUTPUT_DIR = INPUT_DIR / "cleaned_behavior_survey"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MAIN = OUTPUT_DIR / "main_phase.csv"
OUT_SURVEY = OUTPUT_DIR / "survey_phase.csv"
OUT_MAIN_SUMMARY = OUTPUT_DIR / "main_phase_summary.csv"

print("Input folder:", INPUT_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())


Input folder: /Users/jianleguo/Documents/GitHub/Role_Speficifed_Collective_Foraging_Task/data/Pilot_Official
Output folder: /Users/jianleguo/Documents/GitHub/Role_Speficifed_Collective_Foraging_Task/data/Pilot_Official/cleaned_behavior_survey


In [2]:

def first_nonnull(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan


def last_nonnull(s):
    s = s.dropna()
    return s.iloc[-1] if len(s) else np.nan


def clean_strings(df):
    out = df.copy()
    for c in out.select_dtypes(include="object").columns:
        out[c] = out[c].map(lambda x: x.strip() if isinstance(x, str) else x)
    return out


def score_items(row, cols, max_score=None, reverse_cols=None):
    vals = []
    reverse_cols = set(reverse_cols or [])
    for c in cols:
        v = pd.to_numeric(row.get(c, np.nan), errors="coerce")
        if pd.isna(v):
            vals.append(np.nan)
        elif c in reverse_cols and max_score is not None:
            vals.append((max_score + 1) - v)
        else:
            vals.append(v)
    return pd.Series(vals, index=cols, dtype="float")


def flatten_columns(df):
    out = df.copy()
    out.columns = [
        "_".join([str(x) for x in col if str(x) != ""]).strip("_")
        if isinstance(col, tuple) else str(col)
        for col in out.columns
    ]
    return out


In [3]:

files = sorted(INPUT_DIR.glob(INPUT_PATTERN))
if not files:
    raise FileNotFoundError(f"No files matching {INPUT_PATTERN} found in {INPUT_DIR}")

frames = []
for f in files:
    d = pd.read_csv(f, low_memory=False)
    d["source_file"] = f.name
    frames.append(d)

raw = pd.concat(frames, ignore_index=True)
raw = clean_strings(raw)
raw["iso_time_parsed"] = pd.to_datetime(raw.get("iso_time"), errors="coerce", utc=True)
raw = raw.sort_values(["participant_id", "iso_time_parsed"], na_position="last").reset_index(drop=True)

print(f"Read {len(files)} files and {len(raw):,} total rows.")
display(raw["phase"].value_counts(dropna=False).rename("n_rows"))


Read 6 files and 12,767 total rows.


phase
main           11170
observation     1561
survey            24
system            12
Name: n_rows, dtype: int64

In [4]:

main_keep = [
    "source_file", "participant_id", "iso_time", "iso_time_parsed",
    "phase", "phase_index", "event_type", "event_name",
    "actor_type", "role", "action", "key", "rt_ms",
    "repetition", "round", "turn_global", "turn_index_in_round", "move_index_in_turn",
    "current_x", "current_y", "from_x", "from_y", "to_x", "to_y", "dx", "dy", "clamped",
    "success", "reason",
    "forager_x", "forager_y", "security_x", "security_y", "forager_stun_turns",
    "gold_total", "gold_delta", "tile_gold_mine", "tile_mine_type", "tile_x", "tile_y",
    "depletion_status", "chase_status",
    "alien_id", "alien_x", "alien_y", "has_alien", "chased_away",
    "found_alien_count", "found_alien_id", "scan_center_x", "scan_center_y", "scanned_tile_count",
    "map_name", "map_reward_level", "map_risk_level", "map_num",
    "partner_name", "partner_role", "human_role",
    "decay_prob", "mine_type_key", "mine_type_raw", "rng_u",
]
main_keep = [c for c in main_keep if c in raw.columns]

main_phase = raw.loc[raw["phase"].eq("main"), main_keep].copy()
main_phase["main_row_index"] = main_phase.groupby("participant_id").cumcount() + 1
main_phase["action_clean"] = main_phase["action"].combine_first(main_phase["event_name"])
main_phase["is_role_action"] = main_phase["event_type"].eq("role_action")
main_phase["is_human_action"] = main_phase["actor_type"].eq("human")
main_phase["is_agent_action"] = main_phase["actor_type"].eq("agent")
main_phase["success_bool"] = main_phase["success"].eq(1)
main_phase["rt_sec"] = pd.to_numeric(main_phase["rt_ms"], errors="coerce") / 1000

front = [
    "source_file", "participant_id", "main_row_index", "iso_time", "iso_time_parsed",
    "phase", "event_type", "event_name", "action_clean", "actor_type", "role", "action",
    "is_role_action", "is_human_action", "is_agent_action", "success_bool", "rt_ms", "rt_sec",
]
front = [c for c in front if c in main_phase.columns]
main_phase = main_phase[front + [c for c in main_phase.columns if c not in front]]

main_phase.to_csv(OUT_MAIN, index=False)
print(f"Wrote {OUT_MAIN.name}: {main_phase.shape[0]:,} rows x {main_phase.shape[1]} columns")
display(main_phase.head())


Wrote main_phase.csv: 11,170 rows x 70 columns


,source_file,participant_id,main_row_index,iso_time,iso_time_parsed,phase,event_type,event_name,action_clean,actor_type,...,map_reward_level,map_risk_level,map_num,partner_name,partner_role,human_role,decay_prob,mine_type_key,mine_type_raw,rng_u
1,gridgame_clean_Bill_2026-05-12T04-39-47-530Z.csv,Bill,1,2026-05-12T03:37:10.178Z,2026-05-12 03:37:10.178000+00:00,main,phase_event,game_start,game_start,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gridgame_clean_Bill_2026-05-12T04-39-47-530Z.csv,Bill,2,2026-05-12T03:37:12.995Z,2026-05-12 03:37:12.995000+00:00,main,phase_event,game_start,game_start,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gridgame_clean_Bill_2026-05-12T04-39-47-530Z.csv,Bill,3,2026-05-12T03:37:16.268Z,2026-05-12 03:37:16.268000+00:00,main,phase_event,game_start,game_start,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gridgame_clean_Bill_2026-05-12T04-39-47-530Z.csv,Bill,4,2026-05-12T03:37:18.018Z,2026-05-12 03:37:18.018000+00:00,main,phase_event,game_start,game_start,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,gridgame_clean_Bill_2026-05-12T04-39-47-530Z.csv,Bill,5,2026-05-12T03:38:52.266Z,2026-05-12 03:38:52.266000+00:00,main,phase_event,game_start,game_start,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:

g = main_phase.groupby("participant_id", dropna=False)
main_summary = g.agg(
    source_file=("source_file", first_nonnull),
    n_main_rows=("phase", "size"),
    first_main_time=("iso_time_parsed", "min"),
    last_main_time=("iso_time_parsed", "max"),
    n_role_actions=("is_role_action", "sum"),
    n_human_actions=("is_human_action", "sum"),
    n_agent_actions=("is_agent_action", "sum"),
    mean_rt_ms=("rt_ms", "mean"),
    median_rt_ms=("rt_ms", "median"),
    final_gold=("gold_total", last_nonnull),
    total_gold_delta=("gold_delta", "sum"),
    final_repetition=("repetition", last_nonnull),
    final_round=("round", last_nonnull),
).reset_index()

event_counts = pd.crosstab(main_phase["participant_id"], main_phase["event_name"]).add_prefix("n_event_").reset_index()
action_counts = pd.crosstab(main_phase["participant_id"], main_phase["action_clean"]).add_prefix("n_action_").reset_index()

actor_role_counts = pd.crosstab(
    [main_phase["participant_id"]],
    [main_phase["actor_type"], main_phase["role"]],
).reset_index()
actor_role_counts = flatten_columns(actor_role_counts)
actor_role_counts = actor_role_counts.rename(columns={
    c: f"n_{c}" for c in actor_role_counts.columns if c != "participant_id"
})

main_summary = main_summary.merge(event_counts, on="participant_id", how="left")
main_summary = main_summary.merge(action_counts, on="participant_id", how="left")
main_summary = main_summary.merge(actor_role_counts, on="participant_id", how="left")

main_summary.to_csv(OUT_MAIN_SUMMARY, index=False)
print(f"Wrote {OUT_MAIN_SUMMARY.name}: {main_summary.shape[0]:,} rows x {main_summary.shape[1]} columns")
display(main_summary)


Wrote main_phase_summary.csv: 6 rows x 41 columns


,participant_id,source_file,n_main_rows,first_main_time,last_main_time,n_role_actions,n_human_actions,n_agent_actions,mean_rt_ms,median_rt_ms,...,n_action_move_left,n_action_move_right,n_action_move_up,n_action_practice_end,n_action_revive,n_action_scan_chase,n_agent_forager,n_agent_security,n_human_forager,n_human_security
0,Bill,gridgame_clean_Bill_2026-05-12T04-39-47-530Z.csv,1921,2026-05-12 03:37:10.178000+00:00,2026-05-12 04:31:41.050000+00:00,1747,845,868,1622.356086,905.0,...,345,342,336,1,19,85,418,450,400,445
1,Cici Zhang,gridgame_clean_Cici_Zhang_2026-05-12T02-35-53-...,1680,2026-05-12 01:15:16.075000+00:00,2026-05-12 02:18:38.163000+00:00,1539,617,868,2131.050323,930.0,...,327,334,291,1,11,90,418,450,291,326
2,Nishka,gridgame_clean_Nishka_2026-05-12T22-41-56-262Z...,1851,2026-05-12 21:31:27.508000+00:00,2026-05-12 22:31:28.126000+00:00,1699,811,860,1837.147368,904.0,...,303,342,344,1,19,89,410,450,378,433
3,Rumi,gridgame_clean_Rumi_2026-05-12T06-58-46-371Z.csv,1948,2026-05-12 05:52:02.162000+00:00,2026-05-12 06:51:08.398000+00:00,1776,852,867,1699.595971,904.0,...,377,368,337,1,20,107,417,450,407,445
4,Trevor,gridgame_clean_Trevor_2026-05-12T05-25-20-685Z...,1912,2026-05-12 04:15:03.275000+00:00,2026-05-12 05:18:26.904000+00:00,1751,844,864,1783.795119,905.0,...,364,353,305,1,19,97,414,450,403,441
5,carol,gridgame_clean_carol_2026-05-12T22-50-10-613Z.csv,1858,2026-05-12 21:47:07.065000+00:00,2026-05-12 22:42:42.051000+00:00,1699,824,849,1669.066082,906.0,...,314,345,373,1,27,61,399,450,376,448


In [6]:

NFC_ITEMS = [f"nfc_item_{i}" for i in range(1, 19)]
# NFC reverse scoring depends on the exact item wording. Add item names here if needed, e.g. ["nfc_item_3", ...].
NFC_REVERSE_ITEMS = []

BISBAS_ITEMS = [f"bisbas_item_{i}" for i in range(1, 21)]
BISBAS_REVERSE_ITEMS = ["bisbas_item_1", "bisbas_item_18"]
BISBAS_SUBSCALES = {
    "bisbas_bis": [1, 6, 10, 13, 15, 18, 20],
    "bisbas_drive": [2, 7, 9, 17],
    "bisbas_fun_seeking": [4, 8, 12, 16],
    "bisbas_reward_responsiveness": [3, 5, 11, 14, 19],
}

FIVEDCR_ITEMS = [f"fiveDCR_q{i}" for i in range(1, 25)]
FIVEDCR_REVERSE_ITEMS = [f"fiveDCR_q{i}" for i in range(9, 13)]
FIVEDCR_SUBSCALES = {
    "fiveDCR_joyous_exploration": [1, 2, 3, 4],
    "fiveDCR_deprivation_sensitivity": [5, 6, 7, 8],
    "fiveDCR_stress_tolerance_reversed": [9, 10, 11, 12],
    "fiveDCR_thrill_seeking": [13, 14, 15, 16],
    "fiveDCR_social_curiosity_general": [17, 18, 19, 20],
    "fiveDCR_social_curiosity_covert": [21, 22, 23, 24],
}


def get_survey_row(survey_df, survey_name):
    rows = survey_df.loc[survey_df["survey_name"].eq(survey_name)]
    if rows.empty:
        return pd.Series(dtype="object")
    return rows.iloc[-1]


In [7]:

survey_records = []

for pid, p in raw.groupby("participant_id", dropna=False):
    s = p.loc[p["phase"].eq("survey")].copy()
    rec = {
        "participant_id": pid,
        "source_file": first_nonnull(p["source_file"]),
        "n_survey_rows": len(s),
        "survey_names_completed": "|".join(s.get("survey_name", pd.Series(dtype=object)).dropna().astype(str).tolist()),
    }

    final = get_survey_row(s, "final_survey")
    for c in ["age", "birth_year", "sex", "ethnicity", "ethnicity_other_text", "strategy_description", "agent_ranking_order"]:
        rec[c] = final.get(c, np.nan)

    rank_order = final.get("agent_ranking_order", np.nan)
    ranks = [x.strip() for x in rank_order.split("|") if x.strip()] if isinstance(rank_order, str) else []
    for i in range(1, 7):
        rec[f"final_rank_{i}_agent"] = ranks[i - 1] if i <= len(ranks) else np.nan

    nfc = get_survey_row(s, "need_for_cognition")
    nfc_cols = [c for c in NFC_ITEMS if c in raw.columns]
    nfc_raw = score_items(nfc, nfc_cols)
    nfc_scored = score_items(nfc, nfc_cols, max_score=5, reverse_cols=NFC_REVERSE_ITEMS)
    for c, v in nfc_raw.items():
        rec[c] = v
    rec["nfc_raw_mean"] = nfc_raw.mean(skipna=True)
    rec["nfc_raw_sum"] = nfc_raw.sum(skipna=True)
    rec["nfc_scored_mean"] = nfc_scored.mean(skipna=True)
    rec["nfc_scored_sum"] = nfc_scored.sum(skipna=True)

    bis = get_survey_row(s, "bis_bas")
    bis_cols = [c for c in BISBAS_ITEMS if c in raw.columns]
    bis_scored = score_items(bis, bis_cols, max_score=4, reverse_cols=BISBAS_REVERSE_ITEMS)
    for c in bis_cols:
        rec[c] = pd.to_numeric(bis.get(c, np.nan), errors="coerce")
    for scale, items in BISBAS_SUBSCALES.items():
        cols = [f"bisbas_item_{i}" for i in items if f"bisbas_item_{i}" in bis_scored.index]
        rec[f"{scale}_mean"] = bis_scored[cols].mean(skipna=True)
        rec[f"{scale}_sum"] = bis_scored[cols].sum(skipna=True)

    fdc = get_survey_row(s, "five_dcr")
    fdc_cols = [c for c in FIVEDCR_ITEMS if c in raw.columns]
    fdc_scored = score_items(fdc, fdc_cols, max_score=7, reverse_cols=FIVEDCR_REVERSE_ITEMS)
    for c in fdc_cols:
        rec[c] = pd.to_numeric(fdc.get(c, np.nan), errors="coerce")

    fdc_means = []
    for scale, items in FIVEDCR_SUBSCALES.items():
        cols = [f"fiveDCR_q{i}" for i in items if f"fiveDCR_q{i}" in fdc_scored.index]
        m = fdc_scored[cols].mean(skipna=True)
        rec[f"{scale}_mean"] = m
        rec[f"{scale}_sum"] = fdc_scored[cols].sum(skipna=True)
        fdc_means.append(m)
    rec["fiveDCR_overall_mean"] = pd.Series(fdc_means).mean(skipna=True)

    survey_records.append(rec)

survey_phase = pd.DataFrame(survey_records)
survey_phase.to_csv(OUT_SURVEY, index=False)

print(f"Wrote {OUT_SURVEY.name}: {survey_phase.shape[0]:,} rows x {survey_phase.shape[1]} columns")
display(survey_phase)


Wrote survey_phase.csv: 6 rows x 104 columns


,participant_id,source_file,n_survey_rows,survey_names_completed,age,birth_year,sex,ethnicity,ethnicity_other_text,strategy_description,...,fiveDCR_deprivation_sensitivity_sum,fiveDCR_stress_tolerance_reversed_mean,fiveDCR_stress_tolerance_reversed_sum,fiveDCR_thrill_seeking_mean,fiveDCR_thrill_seeking_sum,fiveDCR_social_curiosity_general_mean,fiveDCR_social_curiosity_general_sum,fiveDCR_social_curiosity_covert_mean,fiveDCR_social_curiosity_covert_sum,fiveDCR_overall_mean
0,Bill,gridgame_clean_Bill_2026-05-12T04-39-47-530Z.csv,4,need_for_cognition|bis_bas|five_dcr|final_survey,23.0,2003.0,male,asian,NaN,"Clear spaces as security, stay within 10 steps...",...,16.0,4.25,17.0,3.75,15.0,4.00,16.0,4.75,19.0,4.208333
1,Cici Zhang,gridgame_clean_Cici_Zhang_2026-05-12T02-35-53-...,4,need_for_cognition|bis_bas|five_dcr|final_survey,24.0,2002.0,female,asian,NaN,"If I'm on the yellow side, I will just try to ...",...,16.0,3.75,15.0,4.75,19.0,5.00,20.0,6.25,25.0,4.791667
2,Nishka,gridgame_clean_Nishka_2026-05-12T22-41-56-262Z...,4,need_for_cognition|bis_bas|five_dcr|final_survey,21.0,2005.0,female,asian,NaN,"As a forager, I focused on digging up gold whe...",...,25.0,2.50,10.0,2.00,8.0,6.00,24.0,2.00,8.0,4.083333
3,Rumi,gridgame_clean_Rumi_2026-05-12T06-58-46-371Z.csv,4,need_for_cognition|bis_bas|five_dcr|final_survey,21.0,2005.0,female,asian,NaN,Forager - occasionally explore but mainly dig ...,...,23.0,1.50,6.0,2.25,9.0,5.75,23.0,4.75,19.0,3.791667
4,Trevor,gridgame_clean_Trevor_2026-05-12T05-25-20-685Z...,4,need_for_cognition|bis_bas|five_dcr|final_survey,19.0,2007.0,male,asian,NaN,"When I was security, I tried to stick close to...",...,16.0,5.00,20.0,5.25,21.0,7.00,28.0,6.75,27.0,5.541667
5,carol,gridgame_clean_carol_2026-05-12T22-50-10-613Z.csv,4,need_for_cognition|bis_bas|five_dcr|final_survey,21.0,2005.0,female,asian,NaN,"As a Forager, I would go to all the spots that...",...,19.0,4.75,19.0,2.75,11.0,6.25,25.0,3.50,14.0,4.541667


In [8]:

print("Done.")
print("Main phase CSV:", OUT_MAIN)
print("Survey phase CSV:", OUT_SURVEY)
print("Main phase summary CSV:", OUT_MAIN_SUMMARY)


Done.
Main phase CSV: cleaned_behavior_survey/main_phase.csv
Survey phase CSV: cleaned_behavior_survey/survey_phase.csv
Main phase summary CSV: cleaned_behavior_survey/main_phase_summary.csv
